# 05 — CA16 Ablation Results

## Purpose

CA16 compares bottleneck-only text cross-attention (**CA8**) with an added 16×16 cross-attention block (**CA8+CA16**) in the 32×32 Tiny Avatar DDPM. The research question was whether the added module improves requested-attribute adherence and compositional generalization under a paired, three-training-seed design.

This notebook is CPU-only and uses compact, verified transcriptions of frozen R10–R14 summaries. It does not load models, checkpoints, generated images, raw datasets, or ignored result roots; it does not sample, train, or recompute metrics.

In [ ]:
from __future__ import annotations

import platform
from pathlib import Path

import matplotlib.pyplot as plt


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'src' / 'tiny_avatar_diffusion').is_dir():
            return candidate
    raise RuntimeError('Repository root not found. Run from inside the Tiny Avatar Diffusion repository.')


PROJECT_ROOT = find_project_root()
{'python': platform.python_version(), 'project_root': PROJECT_ROOT.name, 'cuda_used': False}

## Design and hypotheses

Both arms use the 10k generator split (8,388 TRAIN records), a scratch Transformer text encoder, a 32×32 pixel-space DDPM, 1,000 timesteps, AdamW 2e-4, batch 32, AMP FP16, and EMA 0.9999. Three matched initialization/training seeds (2026, 2027, 2028) were trained to 20,000 **successful** optimizer updates. CA8 has 4,723,907 parameters; CA8+CA16 has 4,749,059 (+25,152). Thus the intervention changes both attention placement and parameter count.

R11 selected the 20k milestone using COMP_VAL only. R12 then evaluated the selected EMA checkpoints on ID_TEST and FINAL_OOD with a separately trained real-100k evaluator. Generation seeds are repeated samples within an evaluation; they are not additional independent training runs. KID is R13's primary distribution-quality guardrail; FID is secondary/post-hoc. LPIPS measures within-prompt perceptual variation, not visual quality.

In [ ]:
# Verified transcriptions from the frozen local R11–R14 summaries.
# They are deliberately embedded so this notebook remains useful after a clean clone.
steps = [5_000, 10_000, 15_000, 20_000]
r11_comp_maa = {'CA8': [0.0839, 0.1417, 0.3153, 0.5854], 'CA8+CA16': [0.0909, 0.1462, 0.3889, 0.7374]}
r11_macro_psr = {'CA8': [0.0000, 0.0015, 0.0139, 0.1060], 'CA8+CA16': [0.0000, 0.0000, 0.0282, 0.2656]}
r12 = {
    'CA8': {'ID_TEST MAA': 0.5854, 'FINAL_OOD MAA': 0.5163, 'MacroPSR': 0.2667, 'ID EAM': 0.1118, 'FINAL_OOD EAM': 0.0520},
    'CA8+CA16': {'ID_TEST MAA': 0.7184, 'FINAL_OOD MAA': 0.6758, 'MacroPSR': 0.4880, 'ID EAM': 0.2504, 'FINAL_OOD EAM': 0.1760},
}
r12_paired = {2026: (0.1049, 0.1435, 0.102, 0.052), 2027: (0.1115, 0.1625, 0.398, 0.192), 2028: (0.1826, 0.1725, 0.164, 0.128)}
kid = {'CA8': (0.01756, 0.02041, 0.01827), 'CA8+CA16': (0.01713, 0.01778, 0.01694)}
fid = {'CA8': (53.507, 64.030, 44.231), 'CA8+CA16': (52.282, 60.963, 42.074)}
lpips_delta = {'ID': 0.00030, 'COMP': 0.00204}
efficiency = {
    'B1 seconds/batch': {'CA8': [6.425621, 5.842871, 6.399125], 'CA8+CA16': [6.701372, 6.435503, 7.392571]},
    'B8 seconds/batch': {'CA8': [10.785499, 10.642465, 11.288024], 'CA8+CA16': [11.277465, 11.224221, 23.935962]},
}


## R11 development learning curves

These are COMP_VAL development values used for checkpoint selection, not held-out final results. All six selections were the 20k milestone. The plotted observations do not establish a learning-curve extrapolation or an optimal stopping rule.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), constrained_layout=True)
for arm, values in r11_comp_maa.items():
    axes[0].plot(steps, values, marker='o', label=arm)
for arm, values in r11_macro_psr.items():
    axes[1].plot(steps, values, marker='o', label=arm)
for ax, title, ylabel in zip(axes, ['COMP_VAL adherence', 'COMP_VAL compositional success'], ['MAA', 'MacroPSR']):
    ax.set(title=title, xlabel='successful optimizer steps', ylabel=ylabel, ylim=(0, 0.8))
    ax.grid(alpha=0.25)
    ax.legend()
plt.show()

## R12 selected-checkpoint adherence

R12 uses ID_TEST and FINAL_OOD only. EAM is prompt-level exact attribute match; MacroPSR is the mean over the four FINAL_OOD pair families. CA8+CA16 exceeded CA8 in every paired training seed: FINAL_OOD MAA differences were +0.1435, +0.1625, +0.1725; MacroPSR differences were +0.102, +0.398, +0.164. These are descriptive three-pair observations, not significance tests. FINAL_OOD has been observed and is no longer blind.

In [ ]:
metrics = ['ID_TEST MAA', 'FINAL_OOD MAA', 'MacroPSR', 'ID EAM', 'FINAL_OOD EAM']
x = list(range(len(metrics)))
width = 0.38
fig, ax = plt.subplots(figsize=(10, 3.8), constrained_layout=True)
ax.bar([v - width / 2 for v in x], [r12['CA8'][m] for m in metrics], width, label='CA8')
ax.bar([v + width / 2 for v in x], [r12['CA8+CA16'][m] for m in metrics], width, label='CA8+CA16')
ax.set(xticks=x, xticklabels=metrics, ylim=(0, 0.8), ylabel='score', title='R12 held-out adherence: mean across three training seeds')
ax.legend(); ax.grid(axis='y', alpha=0.25)
plt.show()

for seed, (id_maa, ood_maa, psr, eam) in r12_paired.items():
    print(f'{seed}: ΔID MAA={id_maa:+.4f}, ΔFINAL_OOD MAA={ood_maa:+.4f}, ΔMacroPSR={psr:+.3f}, ΔFINAL_OOD EAM={eam:+.3f}')

## R13 quality guardrail and R14 diversity

Lower KID/FID is better. R13 KID is the prespecified primary quality guardrail; FID is a secondary post-hoc diagnostic and no confidence intervals or significance tests were computed. R14 V3 estimates LPIPS diversity from 100 ID and 100 COMP prompts per arm/training seed, five generation seeds per prompt, and 10 sampled pairs per prompt. The mean CA8+CA16-minus-CA8 LPIPS differences are small and positive (+0.00030 ID; +0.00204 COMP); this is variation, not a quality ranking.

In [ ]:
scopes = ['ID', 'COMP', 'global']
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), constrained_layout=True)
for ax, source, title in zip(axes, (kid, fid), ('R13 KID (primary)', 'R13 FID (secondary/post-hoc)')):
    x = list(range(3)); width = 0.38
    ax.bar([v - width / 2 for v in x], source['CA8'], width, label='CA8')
    ax.bar([v + width / 2 for v in x], source['CA8+CA16'], width, label='CA8+CA16')
    ax.set(xticks=x, xticklabels=scopes, title=title, ylabel='lower is better')
    ax.grid(axis='y', alpha=0.25); ax.legend()
plt.show()
print('R14 V3 LPIPS Δ (CA8+CA16 - CA8):', lpips_delta)

## R14 V3 efficiency

B1 latency is primary and B8 throughput is secondary. These are environment-specific RTX 4060 Laptop GPU/CUDA measurements, not universal costs. The B8 CA8+CA16 seed-2028 median (23.935962 seconds/batch; 0.334225 images/s) is anomalous relative to the other B8 records and is shown, not removed or averaged away.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), constrained_layout=True, sharey=False)
for ax, (label, values) in zip(axes, efficiency.items()):
    for arm, series in values.items():
        ax.plot([2026, 2027, 2028], series, marker='o', label=arm)
    ax.set(title=label, xlabel='training seed', ylabel='median seconds per batch', xticks=[2026, 2027, 2028])
    ax.grid(alpha=0.25); ax.legend()
plt.show()

## Diagnostics, interpretation, and limits

`rimless_glasses` occurred 321 times in 10k TRAIN and had six observed face-shape partners; `pale_pink` occurred 774 times. In contrast, `no_glasses` occurred 4,266 times. The evaluator's frozen real-ceiling audit records a glasses class-4-to-class-5 confusion in both domains and identifies face shape as its relatively weakest head. These are measurement and data-context limitations, not demonstrated causes of a model error.

Matched OOD-to-ID controls are secondary diagnostics: they hold the two target primitive values where matched prompts exist, but contexts, primitive difficulty, evaluator behavior, and finite prompt panels remain entangled. The evidence supports a protocol-specific observed CA8+CA16 advantage, not a placement-only causal conclusion. It does not answer 100k training, balanced sampling, more seeds, alternative capacity controls, H7 continuation to 40k, or native 64×64/single-model multi-resolution support. Those remain future work.

## Evidence and reproduction

The source of the embedded values is the frozen local evidence archive: R11 `results/r11_checkpoint_selection_reproducible_tokenfix_batched_v2/r11_summary.json`; R12 `results/r12_selected_checkpoint_evaluation_batched_v1/<run>/r12_summary.json`; R13 `results/r13_quality_guardrail_batched_v1/<run>/r13_summary.json` and `results/r13_fid_secondary_v1/r13_fid_secondary_summary.json`; and R14 `results/r14_diversity_efficiency_b8_v3/diversity/R14_COMPLETE.json` plus `efficiency/r14_efficiency_summary.json`. Their tracked execution protocols are `data/processed/ca16_r12_eval_v1/r12_execution_protocol.json`, `data/processed/ca16_r13_quality_v1/r13_execution_protocol.json`, and `data/processed/ca16_r14_diversity_efficiency_v1/r14_execution_protocol.json`.

Those result roots and R10 checkpoints are intentionally ignored/local-only, so a clean clone cannot recompute CA16 from this notebook. It can render the summaries and inspect the tracked protocols. For supported V1 inference and clean-clone verification, see [README.md](../README.md), [REPRODUCIBILITY.md](../REPRODUCIBILITY.md), and [FINAL_CLOSURE.md](../docs/experiments/ca16_ablation/FINAL_CLOSURE.md).